# Module B3 - Chatbot Intent Classification Model

This notebook builds the machine learning intent classifier for the Smart Retail FAQ Chatbot.

### Pipeline:
1. **Load Intents**: Parse categories, user patterns, and bot responses from `data/intents.json`.
2. **Text Preprocessing**: Lowercase and clean training patterns.
3. **Feature Extraction**: Transform training text patterns into TF-IDF vector representations.
4. **Model Training**: Train a **Logistic Regression** (or Multinomial Naive Bayes) model to classify query intents.
5. **Response Selection**: Package responses for lookup so that predicted intents can be matched with corresponding responses.
6. **Serialization**: Save the complete model package (classifier, vectorizer, response map) to a single pickle file `chatbot_model.pkl`.

In [1]:
import os
import re
import json
import pickle
import random
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

## 1. Setup Configurations & Load Intents

In [2]:
INTENTS_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'intents.json'))
MODEL_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'app', 'models'))
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Intents JSON path: {INTENTS_PATH}")

with open(INTENTS_PATH, 'r') as f:
    intents_data = json.load(f)

print(f"Loaded {len(intents_data['intents'])} intent categories.")

Intents JSON path: d:\ML Projects\Mp-Online-AIML\data\intents.json
Loaded 9 intent categories.


## 2. Preprocess Training Data
We extract pairs of `(pattern_text, tag)` to build our training dataset.

In [3]:
patterns = []
tags = []
response_map = {} # Tag -> List of responses

for intent in intents_data['intents']:
    tag = intent['tag']
    response_map[tag] = intent['responses']
    
    for pattern in intent['patterns']:
        # Simple text cleaning
        cleaned_pattern = pattern.lower()
        cleaned_pattern = re.sub(r'[^a-z0-9\s]', '', cleaned_pattern)
        
        patterns.append(cleaned_pattern)
        tags.append(tag)

print(f"Total training patterns extracted: {len(patterns)}")
print("Sample Pattern-Tag pair:")
print(f" - Pattern: '{patterns[0]}'\n - Tag: '{tags[0]}'")

Total training patterns extracted: 57
Sample Pattern-Tag pair:
 - Pattern: 'hi'
 - Tag: 'greeting'


## 3. Vectorization with TF-IDF

In [4]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train = vectorizer.fit_transform(patterns)
y_train = np.array(tags)

print("TF-IDF matrix shape:", X_train.shape)

TF-IDF matrix shape: (57, 190)


## 4. Train Intent Classification Model

In [5]:
# Multi-class logistic regression
model = LogisticRegression(class_weight='balanced', max_iter=200)
model.fit(X_train, y_train)
print("Chatbot intent training completed successfully.")

Chatbot intent training completed successfully.


## 5. Test Inference Pipeline

In [6]:
def get_bot_response(user_query):
    cleaned_query = user_query.lower()
    cleaned_query = re.sub(r'[^a-z0-9\s]', '', cleaned_query)
    
    query_tfidf = vectorizer.transform([cleaned_query])
    
    # Get prediction and probabilities
    prediction = model.predict(query_tfidf)[0]
    probs = model.predict_proba(query_tfidf)[0]
    max_prob = np.max(probs)
    
    # Confidence threshold fallback
    if max_prob < 0.25:
        return "I'm sorry, I didn't quite catch that. Could you rephrase your question?"
        
    responses = response_map.get(prediction, ["I am here to help, please ask your question."])
    return random.choice(responses)

# Test cases
test_queries = [
    "Hey there!",
    "What time does the store close?",
    "Can I return a shirt?",
    "I want to track my product package",
    "Random word that model cannot understand"
]

for query in test_queries:
    print(f"\nUser: {query}")
    print(f"Bot:  {get_bot_response(query)}")


User: Hey there!
Bot:  Hi there! How can I assist you with your shopping experience today?

User: What time does the store close?
Bot:  I'm sorry, I didn't quite catch that. Could you rephrase your question?

User: Can I return a shirt?
Bot:  I'm sorry, I didn't quite catch that. Could you rephrase your question?

User: I want to track my product package
Bot:  Please check the shipment tracking link in your confirmation email or provide me with your Order ID so I can look it up.

User: Random word that model cannot understand
Bot:  I'm sorry, I didn't quite catch that. Could you rephrase your question?


## 6. Package & Serialize Model

In [7]:
chatbot_pkg_path = os.path.join(MODEL_DIR, 'chatbot_model.pkl')

# Store model, vectorizer, and response map together to simplify API integration
chatbot_package = {
    "model": model,
    "vectorizer": vectorizer,
    "response_map": response_map
}

with open(chatbot_pkg_path, 'wb') as f:
    pickle.dump(chatbot_package, f)

print(f"Chatbot model package saved successfully to: {chatbot_pkg_path}")

Chatbot model package saved successfully to: d:\ML Projects\Mp-Online-AIML\app\models\chatbot_model.pkl
